In [ ]:
# ── Clone repo from GitHub (run once per Colab session) ───────────────────
import os, sys
if not os.path.exists('/content/protosearch'):
    !git clone https://github.com/bjdraper/protosearch.git /content/protosearch
sys.path.insert(0, '/content/protosearch')
os.chdir('/content/protosearch')
!pip install -q -r /content/protosearch/requirements.txt

# Protosearch: Protein Family Survey
End-to-end pipeline: install → embed → cluster → phylogenetics

**Runtime:** GPU (T4) recommended — *Runtime → Change runtime type → T4 GPU*

> **To adapt this notebook for your own protein family:**
> 1. **Step 3** — set `SURVEY_NAME` to a short label for your analysis (e.g. `'kinase_insects'`)
> 2. **Step 4** — replace `REFERENCE_PROBES` with your UniProt accessions and update `PFAM_IDS`
> 3. **Step 8** — upload your proteome `.faa` / `.fasta` to Google Drive, then set `INPUT_FASTAS`

| Section | What it does |
|---|---|
| Setup | Install tools, mount Drive, download probes + HMM |
| Input | Discover and validate your proteome FASTA(s) from Drive |
| Embed & Search | Filter proteome, HMMER, ESM2 embeddings, FAISS KNN |
| Clustering | Leiden communities, t-SNE, sub-clustering |
| Trees & ASR | MAFFT alignment, FastTree, IQ-TREE2 ancestral reconstruction |

In [ ]:
# ── STEP 1 & 2: Install system tools and Python packages ─────────────────────
!apt-get install -qq hmmer mafft fasttree
!pip install -q numpy pandas scipy scikit-learn biopython \
    fair-esm faiss-cpu leidenalg python-igraph openTSNE \
    ete3 logomaker py3Dmol transformers accelerate seaborn pyyaml tqdm

In [ ]:
# ── STEP 3: Set working directory ─────────────────────────────────────────────
import os, sys

# ── USER: short unique label for this analysis run ────────────────────────────
# Used as the Google Drive subfolder name and as a prefix on output files.
# Examples: 'agpat_crustacea', 'kinase_insects', 'p450_nematodes'
SURVEY_NAME = 'my_protein_survey'

USE_DRIVE = True   # False if running locally

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = f'/content/drive/MyDrive/{SURVEY_NAME}'
else:
    PROJECT_ROOT = '/content/protosearch'

os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
sys.path.insert(0, '/content/protosearch')
print('Survey name:      ', SURVEY_NAME)
print('Working directory:', os.getcwd())

In [ ]:
# ── STEP 4: Define your protein family ───────────────────────────────────────
# ── USER: edit the three blocks below for your protein family ────────────────
#
#   REFERENCE_PROBES   — known reference proteins as (id, UniProt_acc, label, colour)
#                        Fetch sequences from https://www.uniprot.org/ (search → accession)
#                        Hex colours are used in t-SNE/clustering plots.
#   PFAM_IDS           — Pfam domain ID(s) for the HMMER prefilter
#                        Find yours at https://www.ebi.ac.uk/interpro/
#   ALPHAFOLD_TARGETS  — subset of probes for AlphaFold structure download (can be {})
#   CATALYTIC_MOTIFS   — regex patterns highlighted in logo plots (can be {})
#
# The values below reproduce the crustacean AGPAT acyltransferase example.
# ──────────────────────────────────────────────────────────────────────────────
import shutil


REFERENCE_PROBES = [
    ('AGPAT1_HUMAN',   'Q99943', 'AGPAT1',             '#D94040'),
    ('AGPAT2_HUMAN',   'O15120', 'AGPAT2',             '#E05C5C'),
    ('AGPAT3_HUMAN',   'Q9NRZ7', 'AGPAT3',             '#E87878'),
    ('AGPAT4_HUMAN',   'Q9NRZ5', 'AGPAT4',             '#F09090'),
    ('AGPAT5_HUMAN',   'Q9NUQ2', 'AGPAT5',             '#F8B0B0'),
    ('LCLAT1_HUMAN',   'Q6UWP7', 'LCLAT1',             '#F57F17'),
    ('LCLAT1_MOUSE',   'Q6NVG1', 'LCLAT1 (mouse)',     '#E65100'),
    ('LCLAT1_DANRE',   'Q6NYV8', 'LCLAT1 (zebrafish)', '#FB8C00'),
    ('GPAT4_HUMAN',    'Q86UL3', 'GPAT4',              '#00695C'),
    ('GPAT3_HUMAN',    'Q53EU6', 'GPAT3',              '#2E7D32'),
    ('GPAT1_MOUSE',    'Q61586', 'GPAT1 (mouse)',      '#004D40'),
    ('GPAT2_MOUSE',    'Q8BYM8', 'GPAT2 (mouse)',      '#1B5E20'),
    ('AGPAT2_MOUSE',   'Q8K3K7', 'AGPAT2 (mouse)',     '#B71C1C'),
    ('LPCAT2_HUMAN',   'Q7L5N7', 'LPCAT2',             '#1565C0'),
    ('PlsC_ECOLI',     'P26647', 'PlsC (E.coli)',      '#388E3C'),
    ('PlsC_BACSU',     'O34478', 'PlsC (B.sub)',       '#33691E'),
    ('PlsC_THAPS',     'B8BZR1', 'PlsC (diatom)',      '#558B2F'),
    ('Tafazzin_HUMAN', 'Q16635', 'Tafazzin [ctrl]',   '#4A148C'),
    ('LPCAT3_HUMAN',   'Q6P1A2', 'LPCAT3 [ctrl]',     '#B0BEC5'),
    ('MBOAT7_HUMAN',   'Q96N66', 'MBOAT7 [ctrl]',     '#90A4AE'),
]

PFAM_IDS = ['PF01553']   # acyltransferase domain

MIN_LENGTH = 200
MAX_LENGTH = 500

ALPHAFOLD_TARGETS = {
    'AGPAT1_HUMAN':   'Q99943',
    'AGPAT2_HUMAN':   'O15120',
    'AGPAT3_HUMAN':   'Q9NRZ7',
    'AGPAT4_HUMAN':   'Q9NRZ5',
    'AGPAT5_HUMAN':   'Q9NUQ2',
    'LCLAT1_HUMAN':   'Q6UWP7',
    'GPAT4_HUMAN':    'Q86UL3',
    'GPAT3_HUMAN':    'Q53EU6',
    'LPCAT2_HUMAN':   'Q7L5N7',
    'Tafazzin_HUMAN': 'Q16635',
}

CATALYTIC_MOTIFS = {
    'HxxD': {'pattern': 'H.{2}D', 'colour': '#FF6B00', 'role': 'catalytic dyad'},
    'FPxG': {'pattern': 'FP.G',   'colour': '#FFD700', 'role': 'acyl-CoA binding'},
    'EGTR': {'pattern': 'EGTR',   'colour': '#00CED1', 'role': 'Block II conserved'},
}

# Copy pre-built config from repo
shutil.copy('/content/protosearch/config.yaml', 'config.yaml')

print(f'Survey: {SURVEY_NAME}')
print(f'Reference probes: {len(REFERENCE_PROBES)}')
print(f'Pfam IDs: {PFAM_IDS}')

In [ ]:
# ── STEP 5: Write config.yaml from your inputs ────────────────────────────────
import yaml

config = {
    'paths': {
        'data_dir':    'data',
        'results_dir': 'results',
    },
    'reference_probes': [
        {'id': pid, 'acc': acc, 'label': label, 'colour': colour}
        for pid, acc, label, colour in REFERENCE_PROBES
    ],
    'alphafold_targets': ALPHAFOLD_TARGETS,
    'filter':   {'min_length': MIN_LENGTH, 'max_length': MAX_LENGTH},
    'hmmer':    {'evalue': 1e-5, 'cpu': 4, 'profiles': PFAM_IDS},
    'embedding': {'model': 'esm2_t33_650M_UR50D', 'layer': 33, 'dim': 1280,
                  'batch_size': 32, 'device': 'cuda'},
    'clustering':    {'k_neighbors': 25, 'resolution': 2.0, 'pca_dims': 50, 'tsne_perp': 100},
    'subclustering': {},
    'tree':      {'mafft_flags': '--auto --thread 8 --quiet', 'fasttree_model': 'lg',
                  'iqtree_model': 'LG+G4', 'iqtree_bootstrap': 1000, 'iqtree_threads': 8},
    'structure': {'esmfold_model': 'facebook/esmfold_v1'},
    'catalytic_motifs': CATALYTIC_MOTIFS,
}

with open('config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print('config.yaml written.')
print('You can edit it directly at any time to tune clustering or tree parameters.')

In [ ]:
# ── STEP 6: Download reference probes from UniProt ────────────────────────────
from acyltransferase.config import load_config
from acyltransferase.utils  import fetch_uniprot_sequence, write_fasta

cfg = load_config('config.yaml')

probe_dir = 'data/query_sequences'
os.makedirs(probe_dir, exist_ok=True)
probes_faa = f'{probe_dir}/reference_probes.faa'

if not os.path.exists(probes_faa):
    records = []
    for probe in cfg.reference_probes:
        try:
            _, seq = fetch_uniprot_sequence(probe['acc'])
            records.append((probe['id'], seq))
            print(f"  {probe['id']}: {len(seq)} aa")
        except Exception as e:
            print(f"  {probe['id']}: FAILED — {e}")
    write_fasta(records, probes_faa)
    print(f'Saved {len(records)} probes → {probes_faa}')
else:
    from acyltransferase.utils import read_fasta
    print(f'Probes already present: {len(read_fasta(probes_faa))} sequences')

In [ ]:
# ── STEP 7: Download Pfam HMM profile ────────────────────────────────────────
from acyltransferase.search import download_hmm_profile

hmm_dir = 'data/hmm_profiles'
os.makedirs(hmm_dir, exist_ok=True)

for pfam_id in cfg.hmmer['profiles']:
    out = f'{hmm_dir}/{pfam_id}.hmm'
    if not os.path.exists(out):
        print(f'Downloading {pfam_id} ...')
        download_hmm_profile(pfam_id, out)
    else:
        print(f'{pfam_id}.hmm already present')

!hmmpress {hmm_dir}/*.hmm 2>/dev/null || true
print('Done.')

In [ ]:
# ── STEP 8: Locate your proteome FASTA(s) on Google Drive ────────────────────
#
# Before running this cell:
#   1. Upload your proteome .faa / .fasta file(s) to Google Drive.
#      The notebook expects them under the working folder set in Step 3:
#        MyDrive/{SURVEY_NAME}/data/proteins_filtered/   ← NCBI genome-derived
#        MyDrive/{SURVEY_NAME}/data/crustome_filtered/   ← CRUSTOME-derived
#   2. Set INPUT_FASTAS below using the paths printed by the scanner.
#
# INPUT_FASTAS accepts a mix of:
#   Glob (many files)  → 'data/proteins_filtered/*.faa'
#   Single file        → 'data/proteins_filtered/myspecies.faa'
#   Absolute Drive path→ '/content/drive/MyDrive/my_data/proteome.faa'
# ──────────────────────────────────────────────────────────────────────────────
import glob as _glob
from pathlib import Path
from collections import defaultdict

# ── Scan Drive: group discovered FASTAs by directory ────────────────────────
_search_roots = [PROJECT_ROOT, '/content/drive/MyDrive']
_found = []
for _root in _search_roots:
    _found += _glob.glob(f'{_root}/**/*.faa',   recursive=True)
    _found += _glob.glob(f'{_root}/**/*.fasta', recursive=True)
_found = sorted(set(_found))

# Group by parent directory for a clean summary
_by_dir = defaultdict(list)
for _p in _found:
    _by_dir[str(Path(_p).parent)].append(_p)

if _by_dir:
    print(f'Found {len(_found)} FASTA file(s) across {len(_by_dir)} folder(s):\n')
    for _d, _files in sorted(_by_dir.items()):
        _total_mb = sum(Path(p).stat().st_size for p in _files) / 1e6
        print(f'  {_d}/')
        print(f'    {len(_files)} files  |  {_total_mb:.0f} MB total')
        print(f'    glob: \'{_d}/*.faa\'')
        # Show first 3 filenames as examples
        for _p in _files[:3]:
            print(f'    ├ {Path(_p).name}')
        if len(_files) > 3:
            print(f'    └ … and {len(_files)-3} more')
        print()
else:
    print('No .faa/.fasta files found on Drive.')
    print(f'Expected location: {PROJECT_ROOT}/data/proteins_filtered/')

# ── USER: set the glob patterns / paths for your FASTAs ──────────────────────
# Replace the examples below with globs from the directory listing above.
INPUT_FASTAS = [
    'data/proteins_filtered/*.faa',
    'data/crustome_filtered/*.faa',
]

# Auto-select all discovered files if INPUT_FASTAS is cleared to empty
if not INPUT_FASTAS:
    if not _found:
        raise ValueError('No FASTA files found. Upload your proteome to Drive first.')
    INPUT_FASTAS = _found
    print(f'INPUT_FASTAS not set — using all {len(_found)} discovered files.')

# Expand glob patterns into concrete paths
_resolved = []
for _p in INPUT_FASTAS:
    _matches = sorted(_glob.glob(_p, recursive=True))
    if _matches:
        _resolved += _matches
    else:
        _resolved.append(_p)  # keep as-is; validation below will catch missing files
INPUT_FASTAS = _resolved

# Validate
_missing = [p for p in INPUT_FASTAS if not Path(p).exists()]
if _missing:
    raise FileNotFoundError(
        f'{len(_missing)} path(s) not found — check your Drive paths:\n' +
        '\n'.join(f'  {p}' for p in _missing[:10]) +
        ('\n  ...' if len(_missing) > 10 else '')
    )

# Combine into one working FASTA (idempotent)
from acyltransferase.utils import combine_fastas, write_fasta, read_fasta

os.makedirs('data/proteins_raw', exist_ok=True)
raw_fasta = 'data/proteins_raw/combined_input.faa'

if not os.path.exists(raw_fasta):
    _records = combine_fastas(*INPUT_FASTAS, deduplicate=False)
    write_fasta(_records, raw_fasta)
    print(f'\nCombined {len(INPUT_FASTAS)} files → {raw_fasta}')
    print(f'Total sequences: {len(_records):,}')
else:
    _n = len(read_fasta(raw_fasta))
    print(f'\nCombined FASTA already exists: {raw_fasta}  ({_n:,} sequences)')
    print('Delete this file to re-combine from INPUT_FASTAS.')

In [ ]:
# ── STEP 9a: Filter proteins by length ───────────────────────────────────────
from acyltransferase.utils import read_fasta, write_fasta, filter_by_length, deduplicate_fasta

raw_records = read_fasta(raw_fasta)
filtered    = deduplicate_fasta(filter_by_length(
    raw_records,
    min_len=cfg.filter['min_length'],
    max_len=cfg.filter['max_length'],
))

filtered_faa = 'data/proteins_filtered.faa'
write_fasta(filtered, filtered_faa)
print(f'Raw: {len(raw_records)}  →  After filter+dedup: {len(filtered)}')

In [ ]:
# 2. HMMER prefilter
from acyltransferase.search import run_hmmer

hmm_path  = f"data/hmm_profiles/{cfg.hmmer['profiles'][0]}.hmm"
hits_faa  = run_hmmer(
    fasta_path  = filtered_faa,
    hmm_path    = hmm_path,
    output_dir  = 'data/hmmer_hits',
    evalue      = cfg.hmmer['evalue'],
    cpu         = cfg.hmmer['cpu'],
)

hits = read_fasta(hits_faa)
print(f'HMMER hits: {len(hits)} sequences → {hits_faa}')

In [ ]:
# 3. Embed hits with ESM2
from acyltransferase.embed import embed_sequences, save_embeddings

os.makedirs('data/embeddings', exist_ok=True)
emb_npy  = 'data/embeddings/hits.npy'
emb_ids  = 'data/embeddings/hits_ids.txt'

if not Path(emb_npy).exists():
    print(f'Embedding {len(hits)} sequences on {DEVICE} ...')
    embeddings, ids = embed_sequences(
        hits,
        model_name = cfg.embedding['model'],
        batch_size = cfg.embedding['batch_size'],
        device     = cfg.embedding['device'],
        layer      = cfg.embedding['layer'],
    )
    save_embeddings(embeddings, ids, emb_npy, emb_ids)
    print(f'Saved: {embeddings.shape}')
else:
    print(f'Embeddings already exist: {emb_npy}')

In [ ]:
# Embed reference probes
probes_faa = 'data/query_sequences/reference_probes.faa'
ref_npy    = 'data/embeddings/ref.npy'
ref_ids    = 'data/embeddings/ref_ids.txt'

if not Path(ref_npy).exists():
    probe_records = read_fasta(probes_faa)
    ref_emb, rids = embed_sequences(
        probe_records,
        model_name = cfg.embedding['model'],
        batch_size = 4,  # small batch for reference probes
        device     = cfg.embedding['device'],
        layer      = cfg.embedding['layer'],
    )
    save_embeddings(ref_emb, rids, ref_npy, ref_ids)
    print(f'Reference embeddings: {ref_emb.shape}')

In [ ]:
# 4. Build FAISS KNN index
import numpy as np
from acyltransferase.search import build_knn_index

embeddings = np.load(emb_npy)
ids        = Path(emb_ids).read_text().splitlines()

os.makedirs('data/knn_index', exist_ok=True)
build_knn_index(embeddings, ids,
                index_path  = 'data/knn_index/hits.index',
                id_map_path = 'data/knn_index/id_map.tsv')
print(f'FAISS index built: {len(ids)} vectors')

In [ ]:
# 5. Query: find nearest neighbours for each reference probe
from acyltransferase.search import query_knn

probe_records = read_fasta(probes_faa)
knn_hits = query_knn(
    query_sequences = probe_records,
    index_path      = 'data/knn_index/hits.index',
    id_map_path     = 'data/knn_index/id_map.tsv',
    k               = KNN_K,
    embed_kwargs    = {
        'model_name': cfg.embedding['model'],
        'batch_size': 4,
        'device':     cfg.embedding['device'],
    }
)

knn_hits.to_csv('results/knn_hits.tsv', sep='\t', index=False)
print(knn_hits.head(10))

In [ ]:
# Quick distance distribution plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
for qid, grp in knn_hits.groupby('query_id'):
    ax.plot(grp['rank'], grp['l2_dist'], alpha=0.6, linewidth=1,
            label=qid.split('|')[0])
ax.set_xlabel('KNN rank'); ax.set_ylabel('L2 distance')
ax.set_title('KNN distance per reference probe'); ax.legend(fontsize=6, ncol=4)
plt.tight_layout(); plt.show()

In [ ]:
# Load embeddings
embeddings, ids = load_embeddings(
    ('data/embeddings/hits.npy', 'data/embeddings/hits_ids.txt'),
)
ref_emb, ref_ids = load_embeddings(
    ('data/embeddings/ref.npy', 'data/embeddings/ref_ids.txt'),
)
print(f'Hits: {embeddings.shape}   Refs: {ref_emb.shape}')

In [ ]:
# Run Leiden clustering
from acyltransferase.cluster import run_clustering
import os

os.makedirs('results/clustering', exist_ok=True)

result = run_clustering(
    embeddings     = embeddings,
    ids            = ids,
    ref_embeddings = ref_emb,
    ref_ids        = ref_ids,
    ref_colours    = cfg.probe_colours(),
    ref_labels     = cfg.probe_labels(),
    k_neighbors    = K_NEIGHBORS,
    resolution     = RESOLUTION,
    pca_dims       = PCA_DIMS,
    tsne_perp      = TSNE_PERP,
    tsne_cache     = 'data/embeddings/tsne_cache.npy',
)

result.assignments.to_csv('results/clustering/cluster_assignments.tsv', sep='\t', index=False)
result.summary.to_csv('results/clustering/cluster_summary.tsv', sep='\t', index=False)
print(result.summary.to_string(index=False))

In [ ]:
# t-SNE plot
from acyltransferase.visualize import plot_tsne
from sklearn.decomposition import PCA

# project ref probes into the same t-SNE space (approximate via PCA only)
pca = PCA(n_components=PCA_DIMS).fit(embeddings)
ref_pca = pca.transform(ref_emb)

fig = plot_tsne(
    coords        = result.tsne_coords,
    labels        = result.assignments['label'].tolist(),
    label_colours = result.label_colours,
    title         = 'Leiden clustering (all sequences)',
)
fig.savefig('results/clustering/tsne_clusters.png', dpi=150)
fig.show()

In [ ]:
# Sub-clustering (optional)
if SUBCLUSTER:
    from acyltransferase.cluster import run_clustering
    from acyltransferase.utils   import read_fasta, write_fasta

    sub_ids = result.assignments.loc[
        result.assignments['label'] == SUBCLUSTER, 'protein_id'
    ].tolist()
    print(f'Sub-clustering {SUBCLUSTER}: {len(sub_ids)} sequences')

    sub_result = run_clustering(
        embeddings     = embeddings,
        ids            = ids,
        ref_embeddings = ref_emb,
        ref_ids        = ref_ids,
        ref_colours    = cfg.probe_colours(),
        ref_labels     = cfg.probe_labels(),
        k_neighbors    = K_NEIGHBORS,
        resolution     = SUB_RESOLUTION,
        pca_dims       = PCA_DIMS,
        tsne_perp      = SUB_TSNE_PERP,
        subset_ids     = sub_ids,
    )

    slug = SUBCLUSTER.lower().replace(' ', '_')
    sub_result.assignments.to_csv(
        f'results/clustering/{slug}_subcluster_assignments.tsv', sep='\t', index=False
    )
    sub_result.summary.to_csv(
        f'results/clustering/{slug}_subcluster_summary.tsv', sep='\t', index=False
    )

    fig2 = plot_tsne(
        coords        = sub_result.tsne_coords,
        labels        = sub_result.assignments['label'].tolist(),
        label_colours = sub_result.label_colours,
        title         = f'{SUBCLUSTER} sub-clusters',
    )
    fig2.savefig(f'results/clustering/{slug}_subcluster_tsne.png', dpi=150)
    fig2.show()

    # Export FASTA for tree building
    hits = read_fasta('data/hmmer_hits/proteins_filtered_hmmer.faa')
    hit_ids = set(sub_ids)
    sub_hits = [(rid, seq) for rid, seq in hits if rid in hit_ids]
    write_fasta(sub_hits, f'results/{slug}_hits.faa')
    print(f'Exported {len(sub_hits)} sequences → results/{slug}_hits.faa')

In [ ]:
# 1. Build trees for all clusters
from acyltransferase.tree import build_cluster_trees

cluster_fastas = {c: f'results/{c}_hits.faa' for c in CLUSTERS
                  if Path(f'results/{c}_hits.faa').exists()}

trees = build_cluster_trees(
    cluster_fastas = cluster_fastas,
    output_root    = 'results/trees',
    mafft_flags    = cfg.tree['mafft_flags'],
    fasttree_model = cfg.tree['fasttree_model'],
)

for name, (aln, tree) in trees.items():
    print(f'  {name}: {aln.name}  {tree.name}')

In [ ]:
# 2. IQ-TREE2 ASR (runs ~30 min per cluster on Colab T4)
import pandas as pd
from acyltransferase.tree import iqtree

iqtree_outputs = {}

if RUN_IQTREE:
    for cluster in CLUSTERS:
        aln_path = f'results/trees/{cluster}/{cluster}_aligned.faa'
        if not Path(aln_path).exists():
            print(f'  {cluster}: no alignment found, skipping')
            continue
        print(f'\nRunning IQ-TREE2 for {cluster} ...')
        iqtree_outputs[cluster] = iqtree(
            aligned_path = aln_path,
            output_dir   = f'results/iqtree/{cluster}',
            prefix       = cluster,
            model        = cfg.tree['iqtree_model'],
            bootstrap    = IQTREE_BOOTSTRAP,
            threads      = IQTREE_THREADS,
            asr          = True,
        )
        print(f'  {cluster}: done → {iqtree_outputs[cluster]["treefile"]}')

In [ ]:
# 3. Parse ASR + generate plots per cluster
from acyltransferase.asr       import map_iqtree_nodes, find_key_nodes, parse_state_file, \
                                       consensus_sequence, variable_positions, AA_ORDER
from acyltransferase.visualize import plot_sequence_logos, plot_ancestral_table, \
                                       plot_root_confidence

for cluster in CLUSTERS:
    treefile   = Path(f'results/iqtree/{cluster}/{cluster}.treefile')
    state_file = Path(f'results/iqtree/{cluster}/{cluster}.state')
    if not treefile.exists() or not state_file.exists():
        print(f'{cluster}: IQ-TREE outputs not found, skipping')
        continue

    print(f'\nParsing {cluster} ...')
    plot_dir = Path(f'results/plots/ancestral/{cluster}')
    plot_dir.mkdir(parents=True, exist_ok=True)

    # load sub-cluster assignments if available
    sub_tsv = Path(f'results/clustering/{cluster}_subcluster_assignments.tsv')
    sub_df  = pd.read_csv(sub_tsv, sep='\t') if sub_tsv.exists() else None

    # map IQ-TREE node names
    tree, node_map = map_iqtree_nodes(treefile)

    # find key nodes
    key_nodes = find_key_nodes(tree, node_map, sub_df, REF_IDS)
    key_nodes.to_csv(f'results/iqtree/{cluster}/key_nodes.tsv', sep='\t', index=False)
    print(f'  Key nodes: {len(key_nodes)}')

    # parse .state
    target_nodes = set(key_nodes['iqtree_node'].dropna())
    prob_dict    = parse_state_file(state_file, target_nodes)
    print(f'  Loaded {len(prob_dict)} node probability matrices')

    # consensus sequences
    from acyltransferase.utils import write_fasta
    node_label_map = dict(zip(key_nodes['iqtree_node'], key_nodes['node_label']))
    labelled_probs = {node_label_map.get(k, k): v for k, v in prob_dict.items()}

    write_fasta(
        [(lbl, consensus_sequence(probs)) for lbl, probs in labelled_probs.items()],
        f'results/iqtree/{cluster}/ancestral_sequences.faa',
    )

    # variable positions
    var_pos = variable_positions(labelled_probs, top_n=TOP_VAR_POSITIONS)

    # Plot 1: sequence logos
    fig = plot_sequence_logos(labelled_probs, var_pos, AA_ORDER)
    fig.savefig(plot_dir / '01_sequence_logos.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Plot 2: ancestral AA table
    fig = plot_ancestral_table(labelled_probs, var_pos, AA_ORDER)
    fig.savefig(plot_dir / '02_ancestral_table.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Plot 3: root confidence + diversity
    fig = plot_root_confidence(labelled_probs, AA_ORDER)
    fig.savefig(plot_dir / '03_root_confidence.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

    print(f'  Plots saved → {plot_dir}')